In [ ]:
import pandas as pd

train_url = 'https://raw.githubusercontent.com/jmnwong/NSL-KDD-Dataset/master/KDDTrain%2B.txt'
test_url = 'https://raw.githubusercontent.com/jmnwong/NSL-KDD-Dataset/master/KDDTest%2B.txt'

train_df = pd.read_csv(train_url, header=None)
test_df = pd.read_csv(test_url, header=None)

train_df.shape, test_df.shape

In [ ]:
col_names = [
    "duration","protocol_type","service","flag","src_bytes",
    "dst_bytes","land","wrong_fragment","urgent","hot",
    "num_failed_logins","logged_in","num_compromised","root_shell","su_attempted",
    "num_root","num_file_creations","num_shells","num_access_files","num_outbound_cmds",
    "is_host_login","is_guest_login","count","srv_count","serror_rate",
    "srv_serror_rate","rerror_rate","srv_rerror_rate","same_srv_rate","diff_srv_rate",
    "srv_diff_host_rate","dst_host_count","dst_host_srv_count","dst_host_same_srv_rate",
    "dst_host_diff_srv_rate","dst_host_same_src_port_rate","dst_host_srv_diff_host_rate",
    "dst_host_serror_rate","dst_host_srv_serror_rate","dst_host_rerror_rate",
    "dst_host_srv_rerror_rate","class","difficulty_level"
]

train_df.columns = col_names
test_df.columns = col_names

train_df.head()

,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,class,difficulty_level
0,0,tcp,ftp_data,SF,491,0,0,0,0,0,...,0.17,0.03,0.17,0.00,0.00,0.00,0.05,0.00,normal,20
1,0,udp,other,SF,146,0,0,0,0,0,...,0.00,0.60,0.88,0.00,0.00,0.00,0.00,0.00,normal,15
2,0,tcp,private,S0,0,0,0,0,0,0,...,0.10,0.05,0.00,0.00,1.00,1.00,0.00,0.00,neptune,19
3,0,tcp,http,SF,232,8153,0,0,0,0,...,1.00,0.00,0.03,0.04,0.03,0.01,0.00,0.01,normal,21
4,0,tcp,http,SF,199,420,0,0,0,0,...,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,normal,21


In [ ]:
# 1. Collapse the class column into binary: normal vs attack
train_df['label'] = train_df['class'].apply(lambda x: 'normal' if x == 'normal' else 'attack')
test_df['label'] = test_df['class'].apply(lambda x: 'normal' if x == 'normal' else 'attack')

# 2. One-hot encode the three text columns
# We combine train+test first, encode together, then split back apart —
# this avoids a mismatch if train and test don't have identical category values
categorical_cols = ['protocol_type', 'service', 'flag']

combined = pd.concat([train_df, test_df], keys=['train', 'test'])
combined_encoded = pd.get_dummies(combined, columns=categorical_cols)

train_encoded = combined_encoded.loc['train']
test_encoded = combined_encoded.loc['test']

train_encoded.shape, test_encoded.shape

((125973, 125), (22544, 125))

In [ ]:
# Drop columns that shouldn't be part of the model's input
X_train = train_encoded.drop(columns=['class', 'difficulty_level', 'label'])
X_test = test_encoded.drop(columns=['class', 'difficulty_level', 'label'])

# Convert label to binary: 0 = normal, 1 = attack
y_train = (train_encoded['label'] == 'attack').astype(int)
y_test = (test_encoded['label'] == 'attack').astype(int)

X_train.shape, X_test.shape, y_train.value_counts()

((125973, 122),
 (22544, 122),
 label
 0    67343
 1    58630
 Name: count, dtype: int64)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

print("Test accuracy:", model.score(X_test, y_test))
print("Train accuracy:", model.score(X_train, y_train))

Test accuracy: 0.7650372604684174
Train accuracy: 0.999944432537131


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.65      0.97      0.78      9711
           1       0.97      0.61      0.75     12833

    accuracy                           0.77     22544
   macro avg       0.81      0.79      0.76     22544
weighted avg       0.83      0.77      0.76     22544

[[9445  266]
 [5031 7802]]


In [ ]:
# Rebuild a results table: original attack type + what actually happened + what model predicted
results = test_encoded[['class']].copy()
results['actual'] = y_test
results['predicted'] = y_pred

# Filter to just the missed attacks: real attack (actual=1), model said normal (predicted=0)
missed_attacks = results[(results['actual'] == 1) & (results['predicted'] == 0)]

missed_attacks['class'].value_counts()

,count
class,
guess_passwd,1231
warezmaster,806
processtable,668
mscan,655
apache2,599
snmpguess,331
mailbomb,293
snmpgetattack,178
httptunnel,127


In [ ]:
train_df['class'].value_counts()[['guess_passwd', 'warezmaster']]

,count
class,
guess_passwd,53
warezmaster,20


In [ ]:
test_df['class'].value_counts()[['guess_passwd', 'warezmaster']]

,count
class,
guess_passwd,1231
warezmaster,944


In [ ]:
model_balanced = RandomForestClassifier(random_state=42, class_weight='balanced')
model_balanced.fit(X_train, y_train)

y_pred_balanced = model_balanced.predict(X_test)
print(classification_report(y_test, y_pred_balanced))

              precision    recall  f1-score   support

           0       0.66      0.97      0.79      9711
           1       0.97      0.62      0.76     12833

    accuracy                           0.77     22544
   macro avg       0.81      0.80      0.77     22544
weighted avg       0.84      0.77      0.77     22544



In [ ]:
results_balanced = test_encoded[['class']].copy()
results_balanced['actual'] = y_test
results_balanced['predicted'] = y_pred_balanced

missed_balanced = results_balanced[(results_balanced['actual'] == 1) & (results_balanced['predicted'] == 0)]
missed_balanced['class'].value_counts()[['guess_passwd', 'warezmaster']]

,count
class,
guess_passwd,1231
warezmaster,787


In [ ]:
!pip install imbalanced-learn -q
from imblearn.over_sampling import SMOTE

# We need to oversample based on the DETAILED attack type, not just binary,
# so guess_passwd/warezmaster specifically get boosted — not swamped by neptune's abundance.
# So we'll oversample using the full 'class' column, then collapse back to binary after.

y_train_detailed = train_encoded['class']

smote = SMOTE(random_state=42, k_neighbors=1)  # k_neighbors=3 because some classes have very few real examples
X_train_smote, y_train_detailed_smote = smote.fit_resample(X_train, y_train_detailed)

# Collapse back to binary now that classes are balanced
y_train_smote = (y_train_detailed_smote != 'normal').astype(int)

X_train_smote.shape, y_train_detailed_smote.value_counts()[['guess_passwd', 'warezmaster']]

((1548889, 122),
 class
 guess_passwd    67343
 warezmaster     67343
 Name: count, dtype: int64)

In [ ]:
y_train_detailed.value_counts().sort_values()

,count
class,
spy,2
perl,3
phf,4
multihop,7
ftp_write,8
loadmodule,9
rootkit,10
imap,11
land,18


In [ ]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42, k_neighbors=1)
X_train_smote, y_train_detailed_smote = smote.fit_resample(X_train, y_train_detailed)

y_train_smote = (y_train_detailed_smote != 'normal').astype(int)

y_train_detailed_smote.value_counts()[['guess_passwd', 'warezmaster']]

,count
class,
guess_passwd,67343
warezmaster,67343


In [ ]:
model_smote = RandomForestClassifier(random_state=42)
model_smote.fit(X_train_smote, y_train_smote)

y_pred_smote = model_smote.predict(X_test)
print(classification_report(y_test, y_pred_smote))



              precision    recall  f1-score   support

           0       0.66      0.97      0.78      9711
           1       0.97      0.62      0.75     12833

    accuracy                           0.77     22544
   macro avg       0.81      0.80      0.77     22544
weighted avg       0.83      0.77      0.77     22544



In [ ]:
results_smote = test_encoded[['class']].copy()
results_smote['actual'] = y_test
results_smote['predicted'] = y_pred_smote

missed_smote = results_smote[(results_smote['actual'] == 1) & (results_smote['predicted'] == 0)]
missed_smote['class'].value_counts()[['guess_passwd', 'warezmaster']]

,count
class,
guess_passwd,1231
warezmaster,720


In [ ]:
guess_passwd_train = train_df[train_df['class'] == 'guess_passwd']
guess_passwd_test = test_df[test_df['class'] == 'guess_passwd']
normal_train = train_df[train_df['class'] == 'normal']

# A feature that intuitively should matter a lot for password-guessing attacks
for col in ['num_failed_logins', 'logged_in', 'count', 'duration']:
    print(f"--- {col} ---")
    print("guess_passwd (train):", guess_passwd_train[col].describe()[['mean','min','max']].to_dict())
    print("guess_passwd (test): ", guess_passwd_test[col].describe()[['mean','min','max']].to_dict())
    print("normal (train):      ", normal_train[col].describe()[['mean','min','max']].to_dict())
    print()

--- num_failed_logins ---
guess_passwd (train): {'mean': 1.0566037735849056, 'min': 0.0, 'max': 5.0}
guess_passwd (test):  {'mean': 0.3793663688058489, 'min': 0.0, 'max': 1.0}
normal (train):       {'mean': 0.001380989857891689, 'min': 0.0, 'max': 4.0}

--- logged_in ---
guess_passwd (train): {'mean': 0.018867924528301886, 'min': 0.0, 'max': 1.0}
guess_passwd (test):  {'mean': 0.6181965881397238, 'min': 0.0, 'max': 1.0}
normal (train):       {'mean': 0.7106455013884145, 'min': 0.0, 'max': 1.0}

--- count ---
guess_passwd (train): {'mean': 1.528301886792453, 'min': 1.0, 'max': 3.0}
guess_passwd (test):  {'mean': 1.1039805036555645, 'min': 1.0, 'max': 8.0}
normal (train):       {'mean': 22.517945443475938, 'min': 0.0, 'max': 511.0}

--- duration ---
guess_passwd (train): {'mean': 2.7169811320754715, 'min': 0.0, 'max': 60.0}
guess_passwd (test):  {'mean': 2.8651502843216896, 'min': 0.0, 'max': 60.0}
normal (train):       {'mean': 168.5873958689099, 'min': 0.0, 'max': 40504.0}



In [ ]:
warezmaster_train = train_df[train_df['class'] == 'warezmaster']
warezmaster_test = test_df[test_df['class'] == 'warezmaster']

for col in ['service', 'num_access_files', 'hot', 'src_bytes', 'dst_bytes', 'logged_in', 'count']:
    print(f"--- {col} ---")
    if col == 'service':
        print("warezmaster (train):", warezmaster_train[col].value_counts().head(3).to_dict())
        print("warezmaster (test): ", warezmaster_test[col].value_counts().head(3).to_dict())
    else:
        print("warezmaster (train):", warezmaster_train[col].describe()[['mean','min','max']].to_dict())
        print("warezmaster (test): ", warezmaster_test[col].describe()[['mean','min','max']].to_dict())
    print()

--- service ---
warezmaster (train): {'ftp_data': 18, 'ftp': 2}
warezmaster (test):  {'ftp': 498, 'ftp_data': 446}

--- num_access_files ---
warezmaster (train): {'mean': 0.0, 'min': 0.0, 'max': 0.0}
warezmaster (test):  {'mean': 0.0, 'min': 0.0, 'max': 0.0}

--- hot ---
warezmaster (train): {'mean': 0.9, 'min': 0.0, 'max': 18.0}
warezmaster (test):  {'mean': 1.103813559322034, 'min': 0.0, 'max': 2.0}

--- src_bytes ---
warezmaster (train): {'mean': 49.3, 'min': 0.0, 'max': 950.0}
warezmaster (test):  {'mean': 66756.01906779662, 'min': 0.0, 'max': 283618.0}

--- dst_bytes ---
warezmaster (train): {'mean': 3922087.7, 'min': 197.0, 'max': 5155468.0}
warezmaster (test):  {'mean': 614.8453389830509, 'min': 0.0, 'max': 283618.0}

--- logged_in ---
warezmaster (train): {'mean': 0.1, 'min': 0.0, 'max': 1.0}
warezmaster (test):  {'mean': 0.5296610169491526, 'min': 0.0, 'max': 1.0}

--- count ---
warezmaster (train): {'mean': 1.15, 'min': 1.0, 'max': 3.0}
warezmaster (test):  {'mean': 1.2743644

In [ ]:
import pandas as pd

# Standard NSL-KDD attack-to-family mapping
attack_family_map = {
    # DoS
    'neptune': 'DoS', 'smurf': 'DoS', 'back': 'DoS', 'teardrop': 'DoS',
    'pod': 'DoS', 'land': 'DoS', 'apache2': 'DoS', 'udpstorm': 'DoS',
    'processtable': 'DoS', 'mailbomb': 'DoS','worm': 'DoS',
    # Probe
    'satan': 'Probe', 'ipsweep': 'Probe', 'nmap': 'Probe', 'portsweep': 'Probe',
    'mscan': 'Probe', 'saint': 'Probe',
    # R2L
    'guess_passwd': 'R2L', 'ftp_write': 'R2L', 'imap': 'R2L', 'phf': 'R2L',
    'multihop': 'R2L', 'warezmaster': 'R2L', 'warezclient': 'R2L',
    'spy': 'R2L', 'xlock': 'R2L', 'xsnoop': 'R2L', 'snmpguess': 'R2L',
    'snmpgetattack': 'R2L', 'httptunnel': 'R2L', 'sendmail': 'R2L',
    'named': 'R2L',
    # U2R
    'buffer_overflow': 'U2R', 'loadmodule': 'U2R', 'rootkit': 'U2R',
    'perl': 'U2R', 'sqlattack': 'U2R', 'xterm': 'U2R', 'ps': 'U2R',
    # Normal
    'normal': 'Normal'
}

# Assuming your dataframe 'df' has a column called 'label' with attack names
full_df['family'] = full_df['class'].map(attack_family_map)



In [ ]:
from sklearn.model_selection import train_test_split

# Step 1: separate rows into "known families" (everything except DoS)
# and "held-out family" (only DoS) — DoS must NEVER appear in training
known_df = full_df[full_df['family'] != 'DoS'].copy()
held_out_df = full_df[full_df['family'] == 'DoS'].copy()

# Step 2: split ONLY the known families into train/test
# stratify= keeps the same mix of Normal/Probe/R2L/U2R in both pieces
known_train, known_test = train_test_split(
    known_df,
    test_size=0.2,
    random_state=42,
    stratify=known_df['family']
)

# Step 3: training set = only known_train (DoS is fully excluded, on purpose)
train_df = known_train

# Step 4: test set = known_test PLUS all of the held-out DoS rows
# this is what makes DoS a genuine "never seen before" test
test_df = pd.concat([known_test, held_out_df], ignore_index=True)

# Step 5: sanity check — DoS should be 0 in train, and its full count in test
print("TRAIN set — should have NO DoS:")
print(train_df['family'].value_counts())

print("\nTEST set — should have ALL DoS rows:")
print(test_df['family'].value_counts())

TRAIN set — should have NO DoS:
family
Normal    61643
Probe     11262
R2L        3104
U2R          95
Name: count, dtype: int64

TEST set — should have ALL DoS rows:
family
DoS       53387
Normal    15411
Probe      2815
R2L         776
U2R          24
Name: count, dtype: int64


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder

# Columns we'll actually use as input features (drop label/family/etc columns)
feature_cols = [col for col in full_df.columns if col not in ['class', 'label', 'family', 'difficulty_level']]

# One-hot encode categorical columns, keep numeric ones as-is
categorical_cols = ['protocol_type', 'service', 'flag']

X_train = pd.get_dummies(train_df[feature_cols], columns=categorical_cols)
X_test = pd.get_dummies(test_df[feature_cols], columns=categorical_cols)

# Make sure train and test end up with the exact same columns
# (test might have a service/flag value train never saw, or vice versa)
X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

# Our target: predict the FAMILY (Normal/DoS/Probe/R2L/U2R)
y_train = train_df['family']
y_test = test_df['family']

print(X_train.shape, X_test.shape)

(76104, 122) (72413, 122)


In [ ]:
# Train Random Forest on data that has NEVER seen a DoS example
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)

# Predict on test set (which includes all the held-out DoS rows)
y_pred = clf.predict(X_test)

# Overall report — precision/recall/f1 per family
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred))

In [ ]:
from sklearn.metrics import confusion_matrix
import pandas as pd

labels_order = ['Normal', 'DoS', 'Probe', 'R2L', 'U2R']
cm = confusion_matrix(y_test, y_pred, labels=labels_order)

cm_df = pd.DataFrame(cm, index=[f"actual_{l}" for l in labels_order],
                         columns=[f"predicted_{l}" for l in labels_order])
print(cm_df)